# Student Performance: EDA, Regression and Classification

This notebook follows the complete machine learning workflow required for the Student Performance dataset.

## 1. Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
)

df = pd.read_csv("Day18_19_student_habits_performance.csv")

df.head()

## 2. Understand the Dataset

Check the number of rows and columns, column names, data types, and basic summary statistics.

In [ ]:
print("Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nSummary Statistics:")
display(df.describe(include="all").T)

## 3. Identify Numerical and Categorical Variables

In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Numerical Variables:")
print(numerical_columns)

print("\nCategorical Variables:")
print(categorical_columns)

## 4. Check Missing Values

Identify columns containing missing values and their counts.

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("Missing Values:")
display(missing_values)

print("\nTotal Missing Values:", df.isnull().sum().sum())

## 5. Examine Distributions

The exam score distribution is examined first because it is the regression target and is also used to create the Pass/Fail target.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["exam_score"], kde=True)
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam Score")
plt.ylabel("Number of Students")
plt.show()

## 6. Examine Potential Outliers

Use the IQR method to identify potential outliers in numerical variables. Outliers are identified but not automatically removed because they may represent genuine student observations.

In [ ]:
outlier_counts = {}

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_counts[column] = ((df[column] < lower_bound) | 
                              (df[column] > upper_bound)).sum()

outlier_table = pd.DataFrame.from_dict(
    outlier_counts, orient="index", columns=["Potential_Outliers"]
)

display(outlier_table)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[numerical_columns])
plt.title("Boxplots of Numerical Variables")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Examine Correlations

Correlation is used to identify numerical variables that appear related to `exam_score` and to examine relationships among numerical variables.

In [ ]:
correlation_matrix = df[numerical_columns].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

print("Correlation with exam_score:")
display(
    correlation_matrix["exam_score"]
    .sort_values(ascending=False)
    .to_frame("Correlation")
)

## 8. Examine Relationships with Exam Score

Study hours are examined against exam score because the correlation analysis indicates a strong relationship.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="study_hours_per_day", y="exam_score")
plt.title("Study Hours per Day vs Exam Score")
plt.xlabel("Study Hours per Day")
plt.ylabel("Exam Score")
plt.show()

## 9. Compare Exam Scores Across a Categorical Variable

This checks whether exam scores differ across extracurricular participation groups.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="extracurricular_participation", y="exam_score")
plt.title("Exam Score by Extracurricular Participation")
plt.xlabel("Extracurricular Participation")
plt.ylabel("Exam Score")
plt.show()

## 10. EDA Findings

- `study_hours_per_day` has the strongest positive numerical relationship with `exam_score`.
- `mental_health_rating` also shows a positive relationship with exam score.
- `social_media_hours` and `netflix_hours` show negative relationships with exam score.
- `exercise_frequency` and `sleep_hours` show smaller positive relationships with exam score.
- The dataset contains missing values only in `parental_education_level`; these will be handled during preprocessing.
- Potential outliers are present in some numerical variables, but they are retained because there is no evidence that they are invalid observations.

## 11. Prepare Data for Machine Learning

`student_id` is excluded because it is an identifier rather than a meaningful predictive feature. `exam_score` is the regression target.

Categorical variables are one-hot encoded and missing values are imputed. Numerical missing values, if any, are filled with the median; categorical missing values are filled with the most frequent category.

In [ ]:
X = df.drop(columns=["student_id", "exam_score"])
y = df["exam_score"]

numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

print("Features used:", X.columns.tolist())
print("\nRegression target: exam_score")

## 12. Split the Data into Training and Testing Sets

Use 80% of the data for training and 20% for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 13. Train the Linear Regression Model

In [ ]:
linear_regression = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_regression.fit(X_train, y_train)

y_train_pred = linear_regression.predict(X_train)
y_test_pred = linear_regression.predict(X_test)

## 14. Evaluate the Regression Model

Use Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² score.

In [ ]:
def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2 Score": r2_score(actual, predicted)
    }

regression_results = pd.DataFrame({
    "Training": regression_metrics(y_train, y_train_pred),
    "Testing": regression_metrics(y_test, y_test_pred)
})

display(regression_results)

## 15. Create the Pass/Fail Classification Target

A student is classified as:

- **Pass**: `exam_score >= 50`
- **Fail**: `exam_score < 50`

In [ ]:
df["pass_fail"] = np.where(df["exam_score"] >= 50, "Pass", "Fail")

print(df["pass_fail"].value_counts())

## 16. Prepare and Split the Classification Data

Use the same prepared predictor variables and exclude both `exam_score` and the newly created `pass_fail` target from the features.

In [ ]:
X_class = df.drop(columns=["student_id", "exam_score", "pass_fail"])
y_class = df["pass_fail"]

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_class,
    y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

classification_numerical = X_class.select_dtypes(include=np.number).columns.tolist()
classification_categorical = X_class.select_dtypes(include="object").columns.tolist()

classification_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            classification_numerical
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            classification_categorical
        )
    ]
)

## 17. Train the Classification Model

Logistic Regression is used because Pass/Fail is a binary classification problem.

In [ ]:
logistic_regression = Pipeline(
    steps=[
        ("preprocessor", classification_preprocessor),
        ("model", LogisticRegression(max_iter=2000))
    ]
)

logistic_regression.fit(X_train_class, y_train_class)

y_train_class_pred = logistic_regression.predict(X_train_class)
y_test_class_pred = logistic_regression.predict(X_test_class)

## 18. Evaluate the Classification Model

Evaluate the model using a confusion matrix, accuracy, precision, recall, and F1-score.

In [ ]:
print("Testing Confusion Matrix:")
cm = confusion_matrix(y_test_class, y_test_class_pred)
display(pd.DataFrame(
    cm,
    index=["Actual Fail", "Actual Pass"],
    columns=["Predicted Fail", "Predicted Pass"]
))

classification_results = pd.DataFrame({
    "Training": [
        accuracy_score(y_train_class, y_train_class_pred),
        precision_score(y_train_class, y_train_class_pred, pos_label="Pass"),
        recall_score(y_train_class, y_train_class_pred, pos_label="Pass"),
        f1_score(y_train_class, y_train_class_pred, pos_label="Pass")
    ],
    "Testing": [
        accuracy_score(y_test_class, y_test_class_pred),
        precision_score(y_test_class, y_test_class_pred, pos_label="Pass"),
        recall_score(y_test_class, y_test_class_pred, pos_label="Pass"),
        f1_score(y_test_class, y_test_class_pred, pos_label="Pass")
    ]
}, index=["Accuracy", "Precision", "Recall", "F1-Score"])

display(classification_results)

## 19. Training vs Testing Performance and Model Fit

Compare the training and testing metrics.

For the regression model, similar training and testing R² scores and errors indicate that the model generalizes reasonably well.

For the classification model, similar training and testing accuracy, precision, recall, and F1-score indicate that there is no strong evidence of overfitting.

A large gap between training and testing performance would suggest overfitting, while low performance on both sets would suggest underfitting.

## 20. Most Important Findings

The EDA shows that study time is the strongest numerical factor associated with exam score. Mental health rating has a positive association, while higher social media and Netflix usage have negative associations. Sleep and exercise show smaller positive relationships. Attendance has a positive but relatively weak numerical correlation in this dataset.

The linear regression model provides a strong fit with similar training and testing performance. The classification model also shows very similar training and testing results, suggesting no major overfitting or underfitting.

## 21. Conclusion: Five Meaningful Insights

1. **Study time is the strongest predictor of exam performance.** Students who spend more hours studying per day tend to achieve higher exam scores.

2. **Mental health is positively associated with academic performance.** Higher mental health ratings are associated with higher exam scores in this dataset.

3. **Higher entertainment and social-media usage is associated with lower scores.** Both social media hours and Netflix hours show negative relationships with exam score.

4. **Healthy routines have smaller positive relationships with performance.** Sleep hours and exercise frequency are positively related to exam scores, although their relationships are weaker than study time.

5. **The models generalize reasonably well.** The training and testing results are close for both regression and classification, so there is no strong evidence of overfitting in these models.